# Stacking Dimensions

When your DataArray has multiple dimensions you want clustered together (e.g., variable and region),
pass them as `column_dims`. tsam_xarray stacks them internally and unstacks the results automatically.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook"

da = sample_energy_data(n_days=30)
da_single = da.sel(scenario="low")
print(f"Shape: {dict(da_single.sizes)}")

## Aggregate with multiple column dims

Pass `column_dims=["variable", "region"]` to cluster all variable-region combinations together.

In [ ]:
result = tsam_xarray.aggregate(
    da_single, n_clusters=4, column_dims=["variable", "region"]
)
result.typical_periods.to_dataframe("value").head(10)

## Results are automatically unstacked

The MultiIndex is resolved back to separate dimensions in the results — no manual `.unstack()` needed.

In [ ]:
print("typical_periods dims:", result.typical_periods.dims)
result.typical_periods.sel(variable="solar").plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Typical periods (solar)",
)

This also works on accuracy metrics and the reconstructed time series:

In [ ]:
result.accuracy.rmse.to_dataframe("RMSE")

In [ ]:
result.reconstructed.sel(variable="solar").plotly.line(
    x="time",
    color="region",
    title="Reconstructed (solar)",
)

## Why column_dims?

`column_dims` controls **what gets clustered together**. By including both `variable` and `region`,
all combinations share the same cluster assignments.

If instead you only want to cluster across variables (treating regions independently),
only pass `variable` — region becomes a slice dim automatically:

```python
result = tsam_xarray.aggregate(da_single, n_clusters=4, column_dims="variable")
# Each region gets its own independent clustering
```